# Setup

In [ ]:
import pandas as pd

df_export_prices = pd.read_csv("Commodity_price_index/country_export_commodity_price_indices.csv")
df_currencies = pd.read_csv("all_currencies_standardized_daily_close.csv")

# Countries with floating regime:
countries = ["Albania", "Australia", "Brazil", "Canada", "Chile", "Colombia", "EMU", "Georgia", "Hungary", 
             "Iceland", "India", "Israel", "Japan", "Rep. of Korea", "Madagascar", "Mexico", "Moldova", "New Zealand", 
             "Norway", "Peru", "Poland", "Seychelle", "Somalia", "South Africa", "Sweden", "Thailand", "Uganda", "United Kingdom", "United States", "Uruguay"]

country_currency_map = {
    "Albania": "ALLUSD",
    "Australia": "AUDUSD",
    "Brazil": "BRLUSD",
    "Canada": "CADUSD",
    "Chile": "CLPUSD",
    "Colombia": "COPUSD",
    "EMU": "EURUSD",
    "Georgia": "GELUSD",
    "Hungary": "HUFUSD",
    "Iceland": "ISKUSD",
    "India": "INRUSD",
    "Israel": "ILSUSD",
    "Japan": "JPYUSD",
    "Rep. of Korea": "KRWUSD",
    "Madagascar": "MGAUSD",
    "Mexico": "MXNUSD",
    "Moldova": "MDLUSD",
    "New Zealand": "NZDUSD",
    "Norway": "NOKUSD",
    "Peru": "PENUSD",
    "Poland": "PLNUSD",
    "Seychelle": "SCRUSD",
    "Somalia": "SOSUSD",
    "South Africa": "ZARUSD",
    "Sweden": "SEKUSD",
    "Thailand": "THBUSD",
    "Uganda": "UGXUSD",
    "United Kingdom": "GBPUSD",
    "United States": "USD",
    "Uruguay": "UYUUSD"
}


# Export LP (OLD)

In [ ]:
import pandas as pd
import numpy as np
import localprojections as lp
import matplotlib.pyplot as plt
from scipy.stats import norm
import os
import warnings

# Suppress warnings for a cleaner output
warnings.filterwarnings('ignore')

# --- SETTINGS ---
start_date = "2022-01-01"
end_date = "2025-12-31"

irf_horizon = 90
opt_lags = 5
opt_ci_95 = 0.95
newey_west_lags = 5

# --- 1. Load and Prepare Data ---
print("--- Step 1: Loading and Preparing Data ---")

try:
    df_export_prices = pd.read_csv("country_export_commodity_price_indices.csv", index_col='Date', parse_dates=True)
    df_currencies = pd.read_csv("all_currencies_standardized_daily_close.csv", index_col='timestamp', parse_dates=True)
    print("Files loaded successfully.")
except FileNotFoundError as e:
    print(f"Error loading files: {e}")
    exit()

df_master = df_export_prices.join(df_currencies, how='inner')
df_master = df_master.loc[start_date:end_date]
print(f"Data filtered to the period: {start_date} to {end_date}. Shape: {df_master.shape}")

countries = ["Albania", "Australia", "Brazil", "Canada", "Chile", "Colombia", "EMU", "Georgia", "Hungary",
             "Iceland", "India", "Israel", "Japan", "Rep. of Korea", "Madagascar", "Mexico", "Moldova", "New Zealand",
             "Norway", "Peru", "Poland", "Seychelles", "Somalia", "South Africa", "Sweden", "Thailand", "Uganda",
             "United Kingdom", "United States", "Uruguay"]

country_currency_map = {
    "Albania": "ALLUSD", "Australia": "AUDUSD", "Brazil": "BRLUSD", "Canada": "CADUSD",
    "Chile": "CLPUSD", "Colombia": "COPUSD", "EMU": "EURUSD", "Georgia": "GELUSD",
    "Hungary": "HUFUSD", "Iceland": "ISKUSD", "India": "INRUSD", "Israel": "ILSUSD",
    "Japan": "JPYUSD", "Rep. of Korea": "KRWUSD", "Madagascar": "MGAUSD", "Mexico": "MXNUSD",
    "Moldova": "MDLUSD", "New Zealand": "NZDUSD", "Norway": "NOKUSD", "Peru": "PENUSD",
    "Poland": "PLNUSD", "Seychelles": "SCRUSD", "Somalia": "SOSUSD", "South Africa": "ZARUSD",
    "Sweden": "SEKUSD", "Thailand": "THBUSD", "Uganda": "UGXUSD", "United Kingdom": "GBPUSD",
    "United States": "USD", "Uruguay": "UYUUSD"
}

plot_folder = "LP_results"
if not os.path.exists(plot_folder):
    os.makedirs(plot_folder)
    print(f"Created subfolder: '{plot_folder}'")

# --- 3. Main LP Loop ###
print("\n--- Step 3: Running Projections and Generating Custom Plots ---")

for country in countries:
    if country == "United States":
        continue

    print(f"\nProcessing IRF for: {country}")
    
    currency_ticker = country_currency_map.get(country)
    if not currency_ticker:
        print(f" - Currency ticker not found for {country}. Skipping.")
        continue

    if country not in df_master.columns or currency_ticker not in df_master.columns:
        print(f" - SKIPPING: Missing data for {country}.")
        continue

    df_country = df_master[[country, currency_ticker]].copy()
    df_country['log_com_index'] = np.log(df_country[country])
    df_country['log_currency'] = np.log(df_country[currency_ticker])
    df_country.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_country.dropna(inplace=True)
    
    if df_country.empty:
        print(f" - No valid (finite) data for {country}. Skipping.")
        continue
    
    endog_vars = ['log_com_index', 'log_currency']
    
    try:
        # --- Run the Model to get IRF results ---
        irf = lp.TimeSeriesLP(
            data=df_country[['log_com_index', 'log_currency']],
            Y=endog_vars,
            response=['log_currency'],
            horizon=irf_horizon,
            lags=opt_lags,
            newey_lags=newey_west_lags,
            ci_width=opt_ci_95
        )

        # --- THIS IS THE FIX: Clean the results DataFrame before plotting ---
        # Replace any potential infinities with NaN
        irf.replace([np.inf, -np.inf], np.nan, inplace=True)
        # Drop any rows where the core results are NaN (due to a failed regression at that horizon)
        original_rows = len(irf)
        irf.dropna(subset=['Mean', 'LB', 'UB'], inplace=True)
        cleaned_rows = len(irf)
        if original_rows > cleaned_rows:
            print(f"  -> Note: Dropped {original_rows - cleaned_rows} horizons with non-finite results.")
        # --- END FIX ---

        if not irf.empty:
            irf_subset = irf[(irf['Shock'] == 'log_com_index') & (irf['Response'] == 'log_currency')].copy()
            
            if irf_subset.empty:
                print(" - No valid IRF results for the specified shock/response pair.")
                continue

            z_95 = norm.ppf((1 + 0.95) / 2)
            z_68 = norm.ppf((1 + 0.68) / 2)
            irf_subset['SE'] = (irf_subset['UB'] - irf_subset['Mean']) / z_95
            irf_subset['LB_68'] = irf_subset['Mean'] - z_68 * irf_subset['SE']
            irf_subset['UB_68'] = irf_subset['Mean'] + z_68 * irf_subset['SE']
            
            fig, ax = plt.subplots(figsize=(16, 7))
            horizon_axis = irf_subset['Horizon']

            ax.fill_between(horizon_axis, irf_subset['LB'], irf_subset['UB'], color='skyblue', alpha=0.2, label='95% Confidence Interval')
            ax.fill_between(horizon_axis, irf_subset['LB_68'], irf_subset['UB_68'], color='steelblue', alpha=0.4, label='68% Confidence Interval')
            ax.plot(horizon_axis, irf_subset['Mean'], color='black', lw=3, label='Mean Response')

            ax.axhline(0, color='black', ls='-', lw=1)
            ax.set_title(f"Response of {currency_ticker} to {country} Export Index Shock", fontsize=20)
            ax.set_xlabel('Horizon (Days)', fontsize=16)
            ax.set_ylabel('Response', fontsize=16)
            ax.tick_params(axis='both', labelsize=14)
            ax.set_xlim(left=0, right=irf_horizon)
            ax.grid(True, linestyle='--', alpha=0.6)
            ax.legend()
            
            country_name_clean = country.replace(" ", "_").replace(".", "")
            filename = f"EXPORT_{country_name_clean}_{start_date}_to_{end_date}.png"
            filepath = os.path.join(plot_folder, filename)
            
            plt.savefig(filepath, dpi=300, bbox_inches='tight')
            print(f"  -> Saved plot to: {filepath}")
            
            plt.show()
            plt.close(fig)
        else:
            print(" - IRF DataFrame was empty after estimation/cleaning. Cannot plot.")

    except Exception as e:
        print(f" - An error occurred for {country}: {e}")

print("\n--- Analysis Complete ---")

# Import LP (OLD)

In [ ]:
import pandas as pd
import numpy as np
import localprojections as lp
import matplotlib.pyplot as plt
from scipy.stats import norm
import os
import warnings

# Suppress warnings for a cleaner output
warnings.filterwarnings('ignore')

# --- SETTINGS ---
start_date = "2022-01-01"
end_date = "2025-12-31"

irf_horizon = 90
opt_lags = 5
opt_ci_95 = 0.95
newey_west_lags = 5

# --- 1. Load and Prepare Data ---
print("--- Step 1: Loading and Preparing Data ---")

try:
    df_import_prices = pd.read_csv("country_import_commodity_price_indices.csv", index_col='Date', parse_dates=True)
    df_currencies = pd.read_csv("all_currencies_standardized_daily_close.csv", index_col='timestamp', parse_dates=True)
    print("Files loaded successfully.")
except FileNotFoundError as e:
    print(f"Error loading files: {e}")
    exit()

df_master = df_import_prices.join(df_currencies, how='inner')
df_master = df_master.loc[start_date:end_date]
print(f"Data filtered to the period: {start_date} to {end_date}. Shape: {df_master.shape}")

countries = ["Albania", "Australia", "Brazil", "Canada", "Chile", "Colombia", "EMU", "Georgia", "Hungary",
             "Iceland", "India", "Israel", "Japan", "Rep. of Korea", "Madagascar", "Mexico", "Moldova", "New Zealand",
             "Norway", "Peru", "Poland", "Seychelles", "Somalia", "South Africa", "Sweden", "Thailand", "Uganda",
             "United Kingdom", "United States", "Uruguay"]

country_currency_map = {
    "Albania": "ALLUSD", "Australia": "AUDUSD", "Brazil": "BRLUSD", "Canada": "CADUSD",
    "Chile": "CLPUSD", "Colombia": "COPUSD", "EMU": "EURUSD", "Georgia": "GELUSD",
    "Hungary": "HUFUSD", "Iceland": "ISKUSD", "India": "INRUSD", "Israel": "ILSUSD",
    "Japan": "JPYUSD", "Rep. of Korea": "KRWUSD", "Madagascar": "MGAUSD", "Mexico": "MXNUSD",
    "Moldova": "MDLUSD", "New Zealand": "NZDUSD", "Norway": "NOKUSD", "Peru": "PENUSD",
    "Poland": "PLNUSD", "Seychelles": "SCRUSD", "Somalia": "SOSUSD", "South Africa": "ZARUSD",
    "Sweden": "SEKUSD", "Thailand": "THBUSD", "Uganda": "UGXUSD", "United Kingdom": "GBPUSD",
    "United States": "USD", "Uruguay": "UYUUSD"
}

plot_folder = "LP_results"
if not os.path.exists(plot_folder):
    os.makedirs(plot_folder)
    print(f"Created subfolder: '{plot_folder}'")

# --- 3. Main LP Loop ###
print("\n--- Step 3: Running Projections and Generating Custom Plots ---")

for country in countries:
    if country == "United States":
        continue

    print(f"\nProcessing IRF for: {country}")
    
    currency_ticker = country_currency_map.get(country)
    if not currency_ticker:
        print(f" - Currency ticker not found for {country}. Skipping.")
        continue

    if country not in df_master.columns or currency_ticker not in df_master.columns:
        print(f" - SKIPPING: Missing data for {country}.")
        continue

    df_country = df_master[[country, currency_ticker]].copy()
    df_country['log_com_index'] = np.log(df_country[country])
    df_country['log_currency'] = np.log(df_country[currency_ticker])
    df_country.replace([np.inf, -np.inf], np.nan, inplace=True)
    df_country.dropna(inplace=True)
    
    if df_country.empty:
        print(f" - No valid (finite) data for {country}. Skipping.")
        continue
    
    endog_vars = ['log_com_index', 'log_currency']
    
    try:
        # --- Run the Model to get IRF results ---
        irf = lp.TimeSeriesLP(
            data=df_country[['log_com_index', 'log_currency']],
            Y=endog_vars,
            response=['log_currency'],
            horizon=irf_horizon,
            lags=opt_lags,
            newey_lags=newey_west_lags,
            ci_width=opt_ci_95
        )

        # Replace any potential infinities with NaN
        irf.replace([np.inf, -np.inf], np.nan, inplace=True)
        # Drop any rows where the core results are NaN (due to a failed regression at that horizon)
        original_rows = len(irf)
        irf.dropna(subset=['Mean', 'LB', 'UB'], inplace=True)
        cleaned_rows = len(irf)
        if original_rows > cleaned_rows:
            print(f"  -> Note: Dropped {original_rows - cleaned_rows} horizons with non-finite results.")
        # --- END FIX ---

        if not irf.empty:
            irf_subset = irf[(irf['Shock'] == 'log_com_index') & (irf['Response'] == 'log_currency')].copy()
            
            if irf_subset.empty:
                print(" - No valid IRF results for the specified shock/response pair.")
                continue

            z_95 = norm.ppf((1 + 0.95) / 2)
            z_68 = norm.ppf((1 + 0.68) / 2)
            irf_subset['SE'] = (irf_subset['UB'] - irf_subset['Mean']) / z_95
            irf_subset['LB_68'] = irf_subset['Mean'] - z_68 * irf_subset['SE']
            irf_subset['UB_68'] = irf_subset['Mean'] + z_68 * irf_subset['SE']
            
            fig, ax = plt.subplots(figsize=(16, 7))
            horizon_axis = irf_subset['Horizon']

            ax.fill_between(horizon_axis, irf_subset['LB'], irf_subset['UB'], color='skyblue', alpha=0.2, label='95% Confidence Interval')
            ax.fill_between(horizon_axis, irf_subset['LB_68'], irf_subset['UB_68'], color='steelblue', alpha=0.4, label='68% Confidence Interval')
            ax.plot(horizon_axis, irf_subset['Mean'], color='black', lw=3, label='Mean Response')

            ax.axhline(0, color='black', ls='-', lw=1)
            ax.set_title(f"Response of {currency_ticker} to {country} Import Index Shock", fontsize=20)
            ax.set_xlabel('Horizon (Days)', fontsize=16)
            ax.set_ylabel('Response', fontsize=16)
            ax.tick_params(axis='both', labelsize=14)
            ax.set_xlim(left=0, right=irf_horizon)
            ax.grid(True, linestyle='--', alpha=0.6)
            ax.legend()
            
            country_name_clean = country.replace(" ", "_").replace(".", "")
            filename = f"IMPORT_{country_name_clean}_{start_date}_to_{end_date}.png"
            filepath = os.path.join(plot_folder, filename)
            
            plt.savefig(filepath, dpi=300, bbox_inches='tight')
            print(f"  -> Saved plot to: {filepath}")
            
            plt.show()
            plt.close(fig)
        else:
            print(" - IRF DataFrame was empty after estimation/cleaning. Cannot plot.")

    except Exception as e:
        print(f" - An error occurred for {country}: {e}")

print("\n--- Analysis Complete ---")

# Load data

In [ ]:
import pandas as pd
import numpy as np

# --- CONFIGURATION ---
THRESHOLD_QUANTILE = 0.5

# --- 1. Define a Helper to Load and Clean Immediately ---
def load_and_clean(sheet_name):
    try:
        df = pd.read_excel("data_main.xlsx", sheet_name=sheet_name)
        df = df.set_index("Date")
        df.index = pd.to_datetime(df.index)
        df = df.groupby(df.index).mean()
        return df
    except ValueError as e:
        print(f"Error loading {sheet_name}: {e}")
        return pd.DataFrame()

print("--- Loading and Cleaning Data ---")

# Load Data
df_neer = load_and_clean("neer")
df_fx = load_and_clean("fx_usd")
df_export = load_and_clean("export_indices")
df_import = load_and_clean("import_indices")
df_tot = load_and_clean("terms_of_trade")
df_roro = load_and_clean("roro") # Contains 5 columns
df_vix = load_and_clean("vix")
df_gpr = load_and_clean("gpr")
df_tpu = load_and_clean("tpu")
df_1y = load_and_clean("1y")
df_ird_japan = load_and_clean("ird_japan")
df_ird_usa = load_and_clean("ird_usa")
df_dollar_factor = load_and_clean("dollar_factor")
df_cds = load_and_clean("cds_volatility") 
df_surprise = load_and_clean("econ_surprise") 
df_equity = load_and_clean("equity")

print("Data loaded.")

# --- 2. CREATE INTERACTION VARIABLES ---
print(f"--- Creating Interaction DataFrames ---")

trade_dfs = {'export': df_export, 'import': df_import, 'tot': df_tot}
interactions = {}

# A. Global Uncertainty (VIX, GPR, TPU, Surprise, RoRo)
global_uncertainty = {
    'gpr': df_gpr, 'vix': df_vix, 'tpu': df_tpu, 'surprise': df_surprise
}

for t_name, t_df in trade_dfs.items():
    # 1. Standard Globals
    for u_name, u_df in global_uncertainty.items():
        if not u_df.empty:
            interactions[f"{t_name}_x_{u_name}"] = t_df.multiply(u_df.iloc[:, 0], axis=0)

    # 2. RoRo Specifics (Loop through all 5 columns)
    if not df_roro.empty:
        for col_name in df_roro.columns:
            # Clean column name for variable use (e.g. "z_spreads" -> "roro_z_spreads")
            clean_col = col_name.replace(" ", "_").lower()
            interactions[f"{t_name}_x_roro_{clean_col}"] = t_df.multiply(df_roro[col_name], axis=0)

# B. Panel Uncertainty
panel_uncertainty = {'cds': df_cds, 'ird_usa': df_ird_usa, "ird_japan": df_ird_japan}

for t_name, t_df in trade_dfs.items():
    for u_name, u_df in panel_uncertainty.items():
        if not u_df.empty:
            interactions[f"{t_name}_x_{u_name}"] = t_df.multiply(u_df)

print(f"Generated {len(interactions)} interaction dataframes.")

# --- 3. CREATE THRESHOLD DUMMIES (Binary) ---
print(f"--- Creating Threshold Dummies (Quantile: {THRESHOLD_QUANTILE}) ---")

def create_global_dummy(source_df, col_idx=0):
    if source_df.empty: return pd.DataFrame()
    dummy_df = pd.DataFrame(index=source_df.index)
    series = source_df.iloc[:, col_idx]
    cutoff = series.quantile(THRESHOLD_QUANTILE)
    dummy_df['dummy'] = (series > cutoff).astype(int)
    return dummy_df

def create_panel_dummy(source_df):
    if source_df.empty: return pd.DataFrame()
    dummy_df = pd.DataFrame(index=source_df.index, columns=source_df.columns)
    for col in source_df.columns:
        cutoff = source_df[col].quantile(THRESHOLD_QUANTILE)
        dummy_df[col] = (source_df[col] > cutoff).astype(int)
        mask_nan = source_df[col].isna()
        dummy_df.loc[mask_nan, col] = np.nan
    return dummy_df

# Standard Globals
df_gpr_dummy = create_global_dummy(df_gpr)
df_vix_dummy = create_global_dummy(df_vix)
df_tpu_dummy = create_global_dummy(df_tpu)
df_surprise_dummy = create_global_dummy(df_surprise)

# --- NEW: RORO Dummies Loop ---
# We create a dictionary to hold them, and also individual variables
roro_dummies = {} 

if not df_roro.empty:
    for i, col_name in enumerate(df_roro.columns):
        # Create dummy using specific column index
        dummy_df = create_global_dummy(df_roro, col_idx=i)
        
        # Create a dynamic variable name
        clean_name = col_name.replace("#", "").replace(" ", "").strip().lower() # e.g. "z_spreads"
        var_name = f"df_roro_{clean_name}_dummy"
        
        # Store in globals() so you can use 'df_roro_z_spreads_dummy' directly later
        globals()[var_name] = dummy_df
        roro_dummies[clean_name] = dummy_df
        print(f"  -> Created {var_name}")

# Panels
df_cds_dummy = create_panel_dummy(df_cds)          
df_ird_usa_dummy = create_panel_dummy(df_ird_usa)
df_ird_japan_dummy = create_panel_dummy(df_ird_japan)

print("Created Global Dummies: GPR, VIX, TPU, Surprise + All RoRo Components")
print("Created Panel Dummies: CDS, IRD_USA, IRD_JAPAN")

In [ ]:
# Load functions
def get_country_data(country_name):
    """
    Dynamically builds the dataframe based on SHOCK_VARS and active_controls.
    """
    try:
        df = pd.DataFrame(index=df_neer.index) # Use NEER index as master time index
        
        # --- LOGIC TO SWITCH DEPENDENT VARIABLE SOURCE ---
        if DEPENDENT_VAR_NAME == 'log_neer':
            if country_name in df_neer.columns:
                df[DEPENDENT_VAR_NAME] = np.log(df_neer[country_name])
            else:
                return None
                
        elif DEPENDENT_VAR_NAME == 'log_fx':
            # Skip USA if we are looking at FX against USD (it's always 1)
            if country_name in ["USA", "United States"]:
                return None
            
            if country_name in df_fx.columns:
                # Note on Interpretation: 
                # If df_fx is "Local Currency per USD" (e.g. JPY = 150), 
                # then UP = Depreciation. 
                # If df_fx is "USD per Local" (e.g. AUD = 0.65), 
                # then UP = Appreciation.
                # NEER is usually UP = Appreciation.
                df[DEPENDENT_VAR_NAME] = np.log(df_fx[country_name])
            else:
                return None
        # -------------------------------------------------

        # --- Dynamically load requested shock variables ---
        if 'log_export' in SHOCK_VARS:
            df['log_export'] = np.log(df_export[country_name])
        
        if 'log_import' in SHOCK_VARS:
            df['log_import'] = np.log(df_import[country_name])
            
        if 'terms_of_trade' in SHOCK_VARS:
             df['terms_of_trade'] = np.log(df_tot[country_name])

    except KeyError:
        return None 

    mapped_name = country_name_mapping.get(country_name, country_name)

    # --- Handle Controls ---
    # (This part remains exactly the same as your previous code)
    if 'log_vix' in active_controls: df['log_vix'] = np.log(df_vix.groupby(df_vix.index).mean().iloc[:, 0]) 
    if 'log_gpr' in active_controls: df['log_gpr'] = np.log(df_gpr.groupby(df_gpr.index).mean().iloc[:, 0])
    if 'log_tpu' in active_controls: df['log_tpu'] = np.log(df_tpu.groupby(df_tpu.index).mean().iloc[:, 0])
    if 'log_roro' in active_controls: df['log_roro'] = df_roro.groupby(df_roro.index).mean().iloc[:, 0] 

    if 'yield_1y' in active_controls:
        if mapped_name in df_1y.columns:
            df['yield_1y'] = df_1y.groupby(df_1y.index).mean()[mapped_name] 
        else: return None 

    if 'ird_japan' in active_controls:
         if mapped_name in df_ird_japan.columns:
            df['ird_japan'] = df_ird_japan.groupby(df_ird_japan.index).mean()[mapped_name]
         else: return None
             
    if 'ird_usa' in active_controls:
         if mapped_name in df_ird_usa.columns:
            df['ird_usa'] = df_ird_usa.groupby(df_ird_usa.index).mean()[mapped_name]
         else: return None

    if 'dollar_factor' in active_controls:
        df['dollar_factor'] = df_dollar_factor.groupby(df_dollar_factor.index).mean().iloc[:, 0].cumsum()

    # Filter Dates
    df = df.loc[START_DATE:END_DATE]
    
    # Clean Data (Inf/NaN)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    
    return df

# Define Mapping for Country Names
country_name_mapping = {
    #"Euro area": "Germany", 
    "USA": "United States",
}

# Baseline LP

In [ ]:
import pandas as pd
import numpy as np
import localprojections as lp
import matplotlib.pyplot as plt
from scipy.stats import norm
import os
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# --- 1. CONFIGURATION & SETTINGS ---
START_DATE = "2010-01-01"
END_DATE = "2025-06-30"

# Model Parameters
IRF_HORIZON = 86
LAGS = 5
NEWEY_LAGS = 5
CI_WIDTH = 0.90


# --- 2. VARIABLE SELECTION ---
# Change this to 'log_fx' to use bilateral USD rates
# Change this to 'log_neer' to use effective exchange rates
DEPENDENT_VAR_NAME = 'log_fx'

# *** NEW: Define a LIST of shocks you want to test ***
SHOCK_VARS = ['log_export'] 

# LIST OF CONTROLS
#active_controls = ["log_gpr", "log_vix", "log_roro", "yield_1y", "ird_usa", "dollar_factor"] 
active_controls = ["dollar_factor", "log_gpr", "log_vix", "log_roro"] 


# Define Output Folder
OUTPUT_FOLDER = f"LP_results_{DEPENDENT_VAR_NAME}"
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

# Define Mapping for Country Names
country_name_mapping = {
    #"Euro area": "Germany", 
    "USA": "United States",
}

# --- 3. HELPER FUNCTION TO GET DATA ---
def get_country_data(country_name):
    """
    Dynamically builds the dataframe based on SHOCK_VARS and active_controls.
    """
    try:
        df = pd.DataFrame(index=df_neer.index) # Use NEER index as master time index
        
        # --- LOGIC TO SWITCH DEPENDENT VARIABLE SOURCE ---
        if DEPENDENT_VAR_NAME == 'log_neer':
            if country_name in df_neer.columns:
                df[DEPENDENT_VAR_NAME] = np.log(df_neer[country_name])
            else:
                return None
                
        elif DEPENDENT_VAR_NAME == 'log_fx':
            # Skip USA if we are looking at FX against USD (it's always 1)
            if country_name in ["USA", "United States"]:
                return None
            
            if country_name in df_fx.columns:
                # Note on Interpretation: 
                # If df_fx is "Local Currency per USD" (e.g. JPY = 150), 
                # then UP = Depreciation. 
                # If df_fx is "USD per Local" (e.g. AUD = 0.65), 
                # then UP = Appreciation.
                # NEER is usually UP = Appreciation.
                df[DEPENDENT_VAR_NAME] = np.log(df_fx[country_name])
            else:
                return None
        # -------------------------------------------------

        # --- Dynamically load requested shock variables ---
        if 'log_export' in SHOCK_VARS:
            df['log_export'] = np.log(df_export[country_name])
        
        if 'log_import' in SHOCK_VARS:
            df['log_import'] = np.log(df_import[country_name])
            
        if 'terms_of_trade' in SHOCK_VARS:
             df['terms_of_trade'] = np.log(df_tot[country_name])

    except KeyError:
        return None 

    mapped_name = country_name_mapping.get(country_name, country_name)

    # --- Handle Controls ---
    # (This part remains exactly the same as your previous code)
    if 'log_vix' in active_controls: df['log_vix'] = np.log(df_vix.groupby(df_vix.index).mean().iloc[:, 0]) 
    if 'log_gpr' in active_controls: df['log_gpr'] = np.log(df_gpr.groupby(df_gpr.index).mean().iloc[:, 0])
    if 'log_tpu' in active_controls: df['log_tpu'] = np.log(df_tpu.groupby(df_tpu.index).mean().iloc[:, 0])
    if 'log_roro' in active_controls: df['log_roro'] = df_roro.groupby(df_roro.index).mean().iloc[:, 0] 

    if 'yield_1y' in active_controls:
        if mapped_name in df_1y.columns:
            df['yield_1y'] = df_1y.groupby(df_1y.index).mean()[mapped_name] 
        else: return None 

    if 'ird_japan' in active_controls:
         if mapped_name in df_ird_japan.columns:
            df['ird_japan'] = df_ird_japan.groupby(df_ird_japan.index).mean()[mapped_name]
         else: return None
             
    if 'ird_usa' in active_controls:
         if mapped_name in df_ird_usa.columns:
            df['ird_usa'] = df_ird_usa.groupby(df_ird_usa.index).mean()[mapped_name]
         else: return None

    if 'dollar_factor' in active_controls:
        df['dollar_factor'] = df_dollar_factor.groupby(df_dollar_factor.index).mean().iloc[:, 0].cumsum()

    # Filter Dates
    df = df.loc[START_DATE:END_DATE]
    
    # Clean Data (Inf/NaN)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    
    return df

# --- 4. MAIN LOOP ---
print(f"--- Starting LP Analysis ---")
print(f"Shocks: {SHOCK_VARS}")
print(f"Controls: {active_controls}")

countries_to_run = df_neer.columns

for country in countries_to_run:
    
    df_reg = get_country_data(country)
    
    if df_reg is None or df_reg.empty:
        print(f"SKIPPING {country}: Missing data.")
        continue
        
    print(f"Processing: {country} (Obs: {len(df_reg)})")

    Y_vars = active_controls + SHOCK_VARS + [DEPENDENT_VAR_NAME]
    
    try:
        # --- RUN LOCAL PROJECTION (Once per country) ---
        irf = lp.TimeSeriesLP(
            data=df_reg,
            Y=Y_vars,
            response=[DEPENDENT_VAR_NAME],
            horizon=IRF_HORIZON,
            lags=LAGS,
            newey_lags=NEWEY_LAGS,
            ci_width=0.95
        )

        # --- CLEAN OUTPUT ---
        irf.replace([np.inf, -np.inf], np.nan, inplace=True)
        irf.dropna(subset=['Mean', 'LB', 'UB'], inplace=True)

        if not irf.empty:
            
            # --- NEW: Loop through EACH shock variable to plot separately ---
            for specific_shock in SHOCK_VARS:
                
                irf_subset = irf[(irf['Shock'] == specific_shock) & (irf['Response'] == DEPENDENT_VAR_NAME)].copy()
                
                if irf_subset.empty:
                    continue

                # --- PLOTTING ---
                z_95 = norm.ppf((1 + 0.95) / 2)
                z_68 = norm.ppf((1 + 0.68) / 2)
                
                irf_subset['SE'] = (irf_subset['UB'] - irf_subset['Mean']) / z_95
                irf_subset['LB_68'] = irf_subset['Mean'] - z_68 * irf_subset['SE']
                irf_subset['UB_68'] = irf_subset['Mean'] + z_68 * irf_subset['SE']
                
                fig, ax = plt.subplots(figsize=(10, 6))
                horizon_axis = irf_subset['Horizon']

                ax.fill_between(horizon_axis, irf_subset['LB'], irf_subset['UB'], color='skyblue', alpha=0.2, label='95% CI')
                ax.fill_between(horizon_axis, irf_subset['LB_68'], irf_subset['UB_68'], color='steelblue', alpha=0.4, label='68% CI')
                ax.plot(horizon_axis, irf_subset['Mean'], color='black', lw=2, label='Mean Response')

                ax.axhline(0, color='black', ls='-', lw=0.8)
                
                # Dynamic Title
                ax.set_title(f"{country}: {DEPENDENT_VAR_NAME} Response to {specific_shock}", fontsize=16)
                ax.set_xlabel('Horizon (Days)', fontsize=12)
                ax.set_ylabel(f'{DEPENDENT_VAR_NAME} Response', fontsize=12)
                ax.set_xlim(left=0, right=IRF_HORIZON)
                ax.grid(True, linestyle='--', alpha=0.6)
                ax.legend(loc='best')

                # --- SAVING (Unique filename for each shock) ---
                ctrl_str = "_".join(active_controls) if active_controls else "no_controls"
                clean_country = country.replace(" ", "")
                
                # Filename now includes the specific shock name
                filename = f"IRF_{clean_country}_{specific_shock}_to_{DEPENDENT_VAR_NAME}_with_{ctrl_str}.png"
                filepath = os.path.join(OUTPUT_FOLDER, filename)
                
                plt.savefig(filepath, dpi=150, bbox_inches='tight')
                plt.close(fig)
            
        else:
            print(f"  -> IRF empty for {country}")

    except Exception as e:
        print(f"  -> Error for {country}: {e}")

print("\n--- All Done ---")

# Panel LP - Loop

In [ ]:
import pandas as pd
import numpy as np
import localprojections as lp
import matplotlib.pyplot as plt
from scipy.stats import norm
import os
import warnings
import traceback

# Suppress warnings
warnings.filterwarnings('ignore')

# --- 1. BATCH CONFIGURATION ---

# A. Date Ranges to Loop Through
# Format: ("Start_Date", "End_Date")
DATE_CONFIGS = [
    ("2010-01-01", "2025-06-30"), # Full sample
    ("2010-01-01", "2019-12-31"), # Pre-Covid
    ("2020-01-01", "2025-06-30"), # Post-Covid
    ("2010-01-01", "2021-12-31"), # Pre Russia Ukraine
    ("2022-02-25", "2025-06-30"), # Post Russia Ukraine
]

# B. Dependent Variables
DEP_VAR_CONFIGS = ['log_neer', 'log_fx']

# C. Control Variable Groups
# Format: ("Name_for_File", [List_of_Variables])
CONTROL_CONFIGS = [
    ("Full_Controls", [ 
        "dollar_factor", 
        "log_gpr",           # Global
        "ird_japan",         # Country-specific
        "roro_liquidity",    # Global component
        "roro_equity",       # Global component
        "roro_spreads"       # Global component
    ]),
]

# (Dates and other settings remain the same)
IRF_HORIZON = 86
LAGS = 5
CI_WIDTH = 0.90
SHOCK_VARS = ['log_export']
SELECTED_COUNTRIES = ["Australia", "Brazil", "Canada", "Chile", "Colombia", "Malaysia", "Mexico", "Peru", "Norway", "South Africa"]
PANEL_NAME = "Commodity_Countries"

# Mapping helper (ensure consistency)
country_name_mapping = {"USA": "United States", "Euro area": "Germany"}

# --- 2. MASTER LOOP ---
for start_date, end_date in DATE_CONFIGS:
    OUTPUT_FOLDER = f"Results/LP_results_PANEL_{start_date}_to_{end_date}"
    if not os.path.exists(OUTPUT_FOLDER): os.makedirs(OUTPUT_FOLDER)
        
    for dep_var_name in DEP_VAR_CONFIGS:
        for control_name, active_controls in CONTROL_CONFIGS:
            
            panel_data_list = []
            available = df_neer.columns
            countries_to_run = [c for c in SELECTED_COUNTRIES if c in available]

            for country in countries_to_run:
                if dep_var_name == 'log_fx' and country in ["USA", "United States"]: continue

                try:
                    df = pd.DataFrame(index=df_neer.index)
                    mapped_name = country_name_mapping.get(country, country)
                    
                    # A. Dependent Variable
                    if dep_var_name == 'log_neer': 
                        df[dep_var_name] = np.log(df_neer[country])
                    elif dep_var_name == 'log_fx':
                        if country in df_fx.columns: df[dep_var_name] = np.log(df_fx[country])
                        else: continue

                    # B. Shock Variable
                    if 'terms_of_trade' in SHOCK_VARS:
                        df['terms_of_trade'] = np.log(df_tot[country])
                    
                    # C. Global Controls (Same for all countries)
                    if 'log_export' in SHOCK_VARS:
                        df['log_export'] = np.log(df_export[country])

                    if 'log_import' in SHOCK_VARS:
                        df['log_import'] = np.log(df_import[country])

                    # RoRo Sub-indices (Liquidity, Equity, Spreads)
                    # Note: Adjust column names "Liquidity", "Equity", "Spreads" to match your Excel exactly
                    roro_map = {'roro_liquidity': 'Liquidity', 'roro_equity': 'Equity', 'roro_spreads': 'Spreads'}
                    for ctrl_key, col_name in roro_map.items():
                        if ctrl_key in active_controls:
                            if col_name in df_roro.columns:
                                df[ctrl_key] = df_roro[col_name]
                            else:
                                # Fallback search if names are slightly different (e.g. "Liquidity Index")
                                matches = [c for c in df_roro.columns if col_name.lower() in c.lower()]
                                if matches: df[ctrl_key] = df_roro[matches[0]]

                    # D. Country-Specific Controls
                    if 'ird_japan' in active_controls:
                        if mapped_name in df_ird_japan.columns:
                            df['ird_japan'] = df_ird_japan[mapped_name]
                        else: continue # Skip country if data missing

                    if 'dollar_factor' in active_controls:
                        if mapped_name in df_dollar_factor.columns: 
                            df['dollar_factor'] = df_dollar_factor[mapped_name]

                    # E. Clean and Filter
                    df = df.loc[start_date:end_date]
                    df.replace([np.inf, -np.inf], np.nan, inplace=True)
                    df.dropna(inplace=True)

                    if not df.empty:
                        df['Country'] = country
                        panel_data_list.append(df)

                except Exception as e:
                    print(f"Error processing {country}: {e}")
                    continue

            if not panel_data_list: continue
            
            df_panel = pd.concat(panel_data_list).reset_index().set_index(['Country', 'Date'])

            # --- 3. RUN PANEL LP ---
            # Identify which controls actually made it into the dataframe
            actual_controls = [c for c in active_controls if c in df_panel.columns]
            Y_vars = actual_controls + SHOCK_VARS + [dep_var_name]
            
            try:
                irf = lp.PanelLP(
                    data=df_panel,
                    Y=Y_vars,
                    response=[dep_var_name],
                    horizon=IRF_HORIZON,
                    lags=LAGS,
                    varcov='kernel',
                    ci_width=CI_WIDTH
                )
                
                # Clean Output
                irf.replace([np.inf, -np.inf], np.nan, inplace=True)
                irf.dropna(subset=['Mean', 'LB', 'UB'], inplace=True)

                # --- 5. PLOTTING ---
                for specific_shock in SHOCK_VARS:
                    
                    irf_subset = irf[(irf['Shock'] == specific_shock) & (irf['Response'] == dep_var_name)].copy()
                    
                    if irf_subset.empty: continue

                    # Calc Stats
                    z_val = norm.ppf((1 + CI_WIDTH) / 2)
                    z_68 = norm.ppf((1 + 0.68) / 2)
                    
                    irf_subset['SE'] = (irf_subset['UB'] - irf_subset['Mean']) / z_val
                    irf_subset['LB_68'] = irf_subset['Mean'] - z_68 * irf_subset['SE']
                    irf_subset['UB_68'] = irf_subset['Mean'] + z_68 * irf_subset['SE']
                    
                    # Plot
                    fig, ax = plt.subplots(figsize=(10, 6))
                    horizon_axis = irf_subset['Horizon']

                    # 90% CI (Based on CI_WIDTH)
                    ax.fill_between(horizon_axis, irf_subset['LB'], irf_subset['UB'], color='skyblue', alpha=0.2, label=f'{int(CI_WIDTH*100)}% CI')
                    # 68% CI
                    ax.fill_between(horizon_axis, irf_subset['LB_68'], irf_subset['UB_68'], color='steelblue', alpha=0.4, label='68% CI')
                    # Mean
                    ax.plot(horizon_axis, irf_subset['Mean'], color='black', lw=2, label='Mean Response')

                    ax.axhline(0, color='black', ls='-', lw=0.8)
                    
                    title_str = f"PANEL ({PANEL_NAME}): {dep_var_name} to {specific_shock}\nControls: {control_name} | {start_date} to {end_date}"
                    #ax.set_title(title_str, fontsize=14)
                    ax.set_xlabel('Horizon (Days)', fontsize=12)
                    #ax.set_ylabel(f'{dep_var_name} Response', fontsize=12)
                    ax.set_xlim(left=0, right=IRF_HORIZON)
                    ax.grid(True, linestyle='--', alpha=0.6)
                    ax.legend(loc='best')

                    # Save
                    filename = f"PANEL_{PANEL_NAME}_{dep_var_name}_shk_{specific_shock}_{control_name}.png"
                    plt.savefig(os.path.join(OUTPUT_FOLDER, filename), bbox_inches='tight')
                    plt.close(fig)
                    print(f"   -> Saved: {filename}")

            except Exception as e:
                print(f"   -> Panel LP Failed: {e}")
                # traceback.print_exc()

print("\n--- Batch Panel Analysis Complete ---")

# Threshold Panel LP

In [ ]:
import pandas as pd
import numpy as np
import localprojections as lp
import matplotlib.pyplot as plt
from scipy.stats import norm
import os
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# --- 1. CONFIGURATION ---
START_DATE = "2010-01-01"
END_DATE = "2025-06-30"
IRF_HORIZON = 86
LAGS = 5
CI_WIDTH = 0.90

# Variables
DEPENDENT_VAR_NAME = 'log_fx' # or 'log_neer'

# *** NEW: Use a LIST for shocks ***
SHOCK_VARS = ['log_export']  
# Example multiple: SHOCK_VARS = ['log_export', 'terms_of_trade']

# Threshold setup
THRESHOLD_NAME = 'gpr'  
THRESHOLD_DF = df_gpr_dummy 

# Controls
active_controls = ["dollar_factor", "log_vix"] 

# --- COUNTRY SELECTION ---
# SELECTED_COUNTRIES = None
SELECTED_COUNTRIES = ["Australia", "Brazil", "Canada", "Chile", "Colombia", "Mexico", "Peru", "Norway", "South Africa"] 
PANEL_NAME = "All" if SELECTED_COUNTRIES is None else "Commodity_Exporters"

# Output Folder
OUTPUT_FOLDER = f"LP_results_PANEL_Threshold_{THRESHOLD_QUANTILE}"
if not os.path.exists(OUTPUT_FOLDER): os.makedirs(OUTPUT_FOLDER)


# --- 2. BUILD PANEL DATASET ---
print(f"--- Building Panel Data for Threshold: {THRESHOLD_NAME} ({PANEL_NAME}) ---")

panel_data_list = []

available_countries = df_neer.columns
if SELECTED_COUNTRIES:
    countries_to_run = [c for c in SELECTED_COUNTRIES if c in available_countries]
else:
    countries_to_run = [c for c in available_countries if c not in ["USA", "United States"]]

for country in countries_to_run:
    
    # Hard rule: Skip USA if using bilateral FX against USD
    if DEPENDENT_VAR_NAME == 'log_fx' and country in ["USA", "United States"]:
        continue

    # A. Get Base Data 
    try:
        df = pd.DataFrame(index=df_neer.index)
        
        # Dynamic Dependent Variable
        if DEPENDENT_VAR_NAME == 'log_neer':
            df[DEPENDENT_VAR_NAME] = np.log(df_neer[country])
        elif DEPENDENT_VAR_NAME == 'log_fx':
            if country in df_fx.columns: df[DEPENDENT_VAR_NAME] = np.log(df_fx[country])
            else: continue

        # --- FIX: Dynamic Shock Variable Loading ---
        if 'log_export' in SHOCK_VARS:
            df['log_export'] = np.log(df_export[country])
        if 'log_import' in SHOCK_VARS:
            df['log_import'] = np.log(df_import[country])
        if 'terms_of_trade' in SHOCK_VARS:
             df['terms_of_trade'] = np.log(df_tot[country])
        
        # B. Add Controls
        if 'log_vix' in active_controls: df['log_vix'] = np.log(df_vix.iloc[:, 0])
        if 'log_gpr' in active_controls: df['log_gpr'] = np.log(df_gpr.iloc[:, 0])
        
        if 'yield_1y' in active_controls: 
            if country == "Euro area": c_map = "Germany"
            else: c_map = country
            if c_map in df_1y.columns: df['yield_1y'] = df_1y[c_map]
            else: continue
            
        if 'dollar_factor' in active_controls: df['dollar_factor'] = df_dollar_factor.iloc[:, 0].cumsum()

        # C. Add the Threshold Dummy
        if 'dummy' in THRESHOLD_DF.columns:
            # It's a global variable (GPR/VIX)
            df['threshold_dummy'] = THRESHOLD_DF['dummy']
        else:
            # It's a panel variable (IRD), so we look for the country name
            # We use the mapping or the direct name
            if country in THRESHOLD_DF.columns:
                df['threshold_dummy'] = THRESHOLD_DF[country]
            else:
                # Handle mapping if needed (e.g., "Euro area" vs "Germany")
                c_map = country_name_mapping.get(country, country)
                if c_map in THRESHOLD_DF.columns:
                    df['threshold_dummy'] = THRESHOLD_DF[c_map]
                else:
                    print(f"Warning: No threshold data for {country}")
                    continue

        # D. Clean
        df = df.loc[START_DATE:END_DATE]
        df.replace([np.inf, -np.inf], np.nan, inplace=True)
        df.dropna(inplace=True)

        # E. Add Country ID
        if not df.empty:
            df['Country'] = country
            panel_data_list.append(df)

    except KeyError:
        continue

if not panel_data_list:
    print("Error: No data available.")
else:
    df_panel = pd.concat(panel_data_list)
    df_panel = df_panel.reset_index().set_index(['Country', 'Date'])
    print(f"Panel Shape: {df_panel.shape}")

    # --- 3. RUN THRESHOLD PANEL LP ---
    print("Running Threshold Panel LP...")
    
    # Include ALL shock vars in the regression
    Y_vars = active_controls + SHOCK_VARS + [DEPENDENT_VAR_NAME]
    
    try:
        irf_on, irf_off = lp.ThresholdPanelLPX(
            data=df_panel,
            Y=Y_vars,
            X=[], 
            threshold_var='threshold_dummy', 
            response=[DEPENDENT_VAR_NAME],
            horizon=IRF_HORIZON,
            lags=LAGS,
            varcov='kernel',
            ci_width=CI_WIDTH
        )

        # --- 4. PLOT (Loop through Shocks) ---
        irf_on.replace([np.inf, -np.inf], np.nan, inplace=True)
        irf_on.dropna(subset=['Mean', 'LB', 'UB'], inplace=True)
        irf_off.replace([np.inf, -np.inf], np.nan, inplace=True)
        irf_off.dropna(subset=['Mean', 'LB', 'UB'], inplace=True)

        # *** LOOP START ***
        for specific_shock in SHOCK_VARS:
            
            sub_on = irf_on[(irf_on['Shock'] == specific_shock) & (irf_on['Response'] == DEPENDENT_VAR_NAME)]
            sub_off = irf_off[(irf_off['Shock'] == specific_shock) & (irf_off['Response'] == DEPENDENT_VAR_NAME)]

            if not sub_on.empty:
                fig, ax = plt.subplots(figsize=(10, 6))
                
                # High State (Red)
                ax.plot(sub_on['Horizon'], sub_on['Mean'], color='red', lw=2, label=f'High {THRESHOLD_NAME}')
                ax.fill_between(sub_on['Horizon'], sub_on['LB'], sub_on['UB'], color='red', alpha=0.1)
                
                # Low State (Blue)
                ax.plot(sub_off['Horizon'], sub_off['Mean'], color='blue', lw=2, label=f'Low {THRESHOLD_NAME}')
                ax.fill_between(sub_off['Horizon'], sub_off['LB'], sub_off['UB'], color='blue', alpha=0.1)

                ax.axhline(0, color='black', ls='-', lw=0.8)
                
                # Dynamic Title
                ax.set_title(f"Panel ({PANEL_NAME}): {DEPENDENT_VAR_NAME} Response to {specific_shock}\nConditioned on {THRESHOLD_NAME}", fontsize=14)
                ax.set_xlabel('Horizon (Days)', fontsize=12)
                ax.set_ylabel(f'{DEPENDENT_VAR_NAME} Response', fontsize=12)
                ax.set_xlim(left=0, right=IRF_HORIZON)
                ax.grid(True, linestyle='--', alpha=0.6)
                ax.legend(loc='best')
                
                # Save with specific shock name
                filename = f"{PANEL_NAME}_{DEPENDENT_VAR_NAME}_THRESH_{THRESHOLD_NAME}_{specific_shock}_w_controls_{active_controls}.png"
                plt.savefig(os.path.join(OUTPUT_FOLDER, filename), bbox_inches='tight')
                plt.close(fig)
                print(f"Saved plot: {filename}")
        # *** LOOP END ***

    except Exception as e:
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

# Threshold Panel LP - Loop

In [ ]:
import pandas as pd
import numpy as np
import localprojections as lp
import matplotlib.pyplot as plt
from scipy.stats import norm
import os
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# --- 1. GLOBAL CONFIGURATION ---
START_DATE = "2010-01-01"
END_DATE = "2025-06-30"
IRF_HORIZON = 66
LAGS = 5
CI_WIDTH = 0.90

# Updated Shocks and Controls to match your requirement
SHOCK_VARS = ['log_export'] 

CONTROL_CONFIGS = [
    ("Base_Controls", [ 
        "dollar_factor", 
        "log_gpr",           # Global
        "ird_japan",         # Country-specific
        "roro_liquidity",    # Global component
        "roro_equity",       # Global component
        "roro_spreads"       # Global component
    ]),
]

# Output Folder (Includes the quantile in the name for clarity)
# Note: Ensure THRESHOLD_QUANTILE variable exists from your loading cell, or hardcode it here for the filename
try:
    q_label = str(THRESHOLD_QUANTILE).replace('.', '')
except NameError:
    q_label = "XX"
    
OUTPUT_FOLDER = f"Results/LP_results_PANEL_Threshold_Q{q_label}"
if not os.path.exists(OUTPUT_FOLDER): os.makedirs(OUTPUT_FOLDER)

# (Thresholds, Mapping, etc.)
THRESHOLD_CONFIGS = [
    ("gpr", df_gpr_dummy),
    ("vix", df_vix_dummy),
    ("roro_equities", df_roro_z_equities_dummy),
    ("roro_spreads", df_roro_z_spreads_dummy),
    ("roro_liquidity", df_roro_z_liquidity_dummy),
    ("tpu", df_tpu_dummy),
    ("surprise", df_surprise_dummy),
    ("cds", df_cds_dummy),
    ("ird_usa", df_ird_usa_dummy),
    ("ird_japan", df_ird_japan_dummy)
]

commodity_exporters = ["Australia", "Brazil", "Canada", "Chile", "Colombia", "Mexico", "Malaysia", "Peru", "Norway", "South Africa"]
COUNTRY_CONFIGS = [("Commodity_Exporters", commodity_exporters)]
DEP_VAR_CONFIGS = ['log_fx', 'log_neer']
country_name_mapping = {"USA": "United States", "Euro area": "Germany"}

# --- 2. MASTER LOOP ---
for panel_name, selected_countries in COUNTRY_CONFIGS:
    for dep_var_name in DEP_VAR_CONFIGS:
        for threshold_name, threshold_df in THRESHOLD_CONFIGS:
            for control_name, active_controls in CONTROL_CONFIGS:
                
                panel_data_list = []
                available = df_neer.columns
                countries_to_run = [c for c in selected_countries if c in available]

                for country in countries_to_run:
                    if dep_var_name == 'log_fx' and country in ["USA", "United States"]: continue

                    try:
                        df = pd.DataFrame(index=df_neer.index)
                        mapped_name = country_name_mapping.get(country, country)
                        
                        # A. Dependent Variable
                        if dep_var_name == 'log_neer': 
                            df[dep_var_name] = np.log(df_neer[country])
                        elif dep_var_name == 'log_fx':
                            if country in df_fx.columns: df[dep_var_name] = np.log(df_fx[country])
                            else: continue

                        # B. Shock Variables (Dynamic Load)
                        for shock in SHOCK_VARS:
                            if shock == 'log_export' and country in df_export.columns:
                                df['log_export'] = np.log(df_export[country])
                            elif shock == 'log_import' and country in df_import.columns:
                                df['log_import'] = np.log(df_import[country])
                            elif shock == 'terms_of_trade' and country in df_tot.columns:
                                df['terms_of_trade'] = np.log(df_tot[country])
                        
                        # C. Global Controls
                        if 'log_vix' in active_controls: 
                            df['log_vix'] = np.log(df_vix.iloc[:, 0])
                        if 'log_gpr' in active_controls:
                            df['log_gpr'] = np.log(df_gpr.iloc[:, 0])
                        
                        # RoRo Sub-indices search logic
                        roro_map = {'roro_liquidity': 'Liquidity', 'roro_equity': 'Equity', 'roro_spreads': 'Spreads'}
                        for ctrl_key, col_name in roro_map.items():
                            if ctrl_key in active_controls:
                                if col_name in df_roro.columns:
                                    df[ctrl_key] = df_roro[col_name]
                                else:
                                    matches = [c for c in df_roro.columns if col_name.lower() in c.lower()]
                                    if matches: df[ctrl_key] = df_roro[matches[0]]

                        # D. Country-Specific Controls
                        if 'ird_japan' in active_controls:
                            if mapped_name in df_ird_japan.columns:
                                df['ird_japan'] = df_ird_japan[mapped_name]
                            else: continue # Skip country if missing control data

                        if 'dollar_factor' in active_controls:
                            if mapped_name in df_dollar_factor.columns: 
                                df['dollar_factor'] = df_dollar_factor[mapped_name]

                        # E. Threshold Dummy Assignment
                        if 'dummy' in threshold_df.columns:
                            df['threshold_dummy'] = threshold_df['dummy']
                        else:
                            # Panel threshold (CDS, IRD, etc.)
                            if mapped_name in threshold_df.columns:
                                df['threshold_dummy'] = threshold_df[mapped_name]
                            else: continue

                        # F. Clean and Filter
                        df = df.loc[START_DATE:END_DATE].replace([np.inf, -np.inf], np.nan).dropna()

                        if not df.empty:
                            df['Country'] = country
                            panel_data_list.append(df)

                    except Exception: continue

                if not panel_data_list: continue

                df_panel = pd.concat(panel_data_list).reset_index().set_index(['Country', 'Date'])
                
                # Identify which controls actually made it into the dataframe
                actual_controls = [c for c in active_controls if c in df_panel.columns]
                Y_vars = actual_controls + SHOCK_VARS + [dep_var_name]
                
                try:
                    irf_on, irf_off = lp.ThresholdPanelLPX(
                        data=df_panel,
                        Y=Y_vars,
                        X=[], 
                        threshold_var='threshold_dummy', 
                        response=[dep_var_name],
                        horizon=IRF_HORIZON,
                        lags=LAGS,
                        varcov='kernel',
                        ci_width=CI_WIDTH
                    )

                    # Clean Results
                    irf_on.replace([np.inf, -np.inf], np.nan, inplace=True)
                    irf_on.dropna(subset=['Mean', 'LB', 'UB'], inplace=True)
                    irf_off.replace([np.inf, -np.inf], np.nan, inplace=True)
                    irf_off.dropna(subset=['Mean', 'LB', 'UB'], inplace=True)

                    # --- C. PLOT ---
                    for specific_shock in SHOCK_VARS:
                        sub_on = irf_on[(irf_on['Shock'] == specific_shock) & (irf_on['Response'] == dep_var_name)]
                        sub_off = irf_off[(irf_off['Shock'] == specific_shock) & (irf_off['Response'] == dep_var_name)]

                        if not sub_on.empty:
                            fig, ax = plt.subplots(figsize=(10, 6))
                            
                            # High State (Red)
                            ax.plot(sub_on['Horizon'], sub_on['Mean'], color='red', lw=2, label=f'High {threshold_name.upper()}')
                            ax.fill_between(sub_on['Horizon'], sub_on['LB'], sub_on['UB'], color='red', alpha=0.1)
                            
                            # Low State (Blue)
                            ax.plot(sub_off['Horizon'], sub_off['Mean'], color='blue', lw=2, label=f'Low {threshold_name.upper()}')
                            ax.fill_between(sub_off['Horizon'], sub_off['LB'], sub_off['UB'], color='blue', alpha=0.1)

                            ax.axhline(0, color='black', ls='-', lw=0.8)
                            
                            # Title
                            #ax.set_title(f"Panel ({panel_name}): {dep_var_name} to {specific_shock}\nCondition: {threshold_name.upper()} | Controls: {control_name}", fontsize=14)
                            ax.set_xlabel('Horizon (Days)', fontsize=12)
                            #ax.set_ylabel(f'{dep_var_name} Response', fontsize=12)
                            ax.set_xlim(left=0, right=IRF_HORIZON)
                            ax.grid(True, linestyle='--', alpha=0.6)
                            ax.legend(loc='best')
                            
                            # Filename
                            filename = f"{panel_name}_{dep_var_name}_THRESH_{threshold_name}_{specific_shock}_{control_name}.png"
                            plt.savefig(os.path.join(OUTPUT_FOLDER, filename), bbox_inches='tight')
                            plt.close(fig)
                            print(f"   -> Saved: {filename}")

                except Exception as e:
                    print(f"   -> [!] Error in model: {e}")
                    # traceback.print_exc()

print("\n--- Batch Threshold Analysis Complete ---")

# Rolling window forecast regression

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from linearmodels import PanelOLS
import os
import warnings

# Suppress warnings
warnings.filterwarnings('ignore')

# --- 1. BATCH CONFIGURATION ---
START_DATE = "2010-01-01"
END_DATE = "2025-06-30"
SELECTED_COUNTRIES = ["Australia", "Brazil", "Canada", "Chile", "Colombia", "Malaysia", "Mexico", "Peru", "Norway", "South Africa"]

# Define your lists of settings to loop through
DEP_VAR_LIST = ['log_neer', 'log_fx']
HORIZON_LIST = [1, 5, 22] 
WINDOW_LIST = [252, 504, 1008]

# Calculates the maximum burn-in period required
MAX_WINDOW_SIZE = max(WINDOW_LIST)

# Control Groups
CONTROL_GROUPS = [
    #("DF & VIX controls", ["dollar_factor", "log_vix"]),
    #("DF, VIX, IRD vs US controls", ["dollar_factor", "log_vix", "ird_usa"]),
    ("No Control", [])
]

# Output
OUTPUT_FOLDER = "Results/Results_Rolling_Regressions"
if not os.path.exists(OUTPUT_FOLDER): os.makedirs(OUTPUT_FOLDER)

# --- 2. DATA PREP FUNCTION ---
def build_panel(dep_var, horizon, controls, countries_list):
    panel_data = []
    available = df_neer.columns
    countries_to_run = [c for c in countries_list if c in available]

    for country in countries_to_run:
        # Skip USA if bilateral FX
        if dep_var == 'log_fx' and country in ["USA", "United States"]: continue
        
        try:
            df = pd.DataFrame(index=df_neer.index)
            
            # 1. Dependent Variable Logic
            if dep_var == 'log_neer': 
                log_y = np.log(df_neer[country])
            elif dep_var == 'log_fx': 
                if country in df_fx.columns: log_y = np.log(df_fx[country])
                else: continue
            elif dep_var == 'terms_of_trade':
                # Load Terms of Trade (Log)
                if country in df_tot.columns: log_y = np.log(df_tot[country])
                else: continue
            else:
                # Fallback or error if variable not recognized
                continue
            
            # 2. Predictor (Export or ToT) ##### CHANGE HERE #####################################
            log_x = np.log(df_export[country])

            # 3. Transformations
            df['X_Predictor'] = log_x.diff()
            df['Y_Target'] = (log_y.shift(-horizon) - log_y)

            # 4. Controls
            if 'log_vix' in controls:
                df['log_vix'] = np.log(df_vix.iloc[:, 0]).diff()
            if 'dollar_factor' in controls:
                df['dollar_factor'] = df_dollar_factor.iloc[:, 0] 
            if 'yield_1y' in controls:
                 c_map = "Germany" if country == "Euro area" else country
                 if c_map in df_1y.columns: df['yield_1y'] = df_1y[c_map].diff()
            if 'ird_usa' in controls:
                 c_map = "Germany" if country == "Euro area" else country
                 if c_map in df_ird_usa.columns: 
                     df['ird_usa'] = df_ird_usa[c_map].diff()

            # 5. Clean and ID
            df = df.loc[START_DATE:END_DATE]
            df.replace([np.inf, -np.inf], np.nan, inplace=True)
            df['Country'] = country
            panel_data.append(df)

        except KeyError:
            continue

    if not panel_data: return None
    
    df_panel = pd.concat(panel_data)
    df_panel = df_panel.set_index(['Country', df_panel.index])
    df_panel.dropna(inplace=True)
    return df_panel

# --- 3. MASTER LOOP ---
total_runs = len(DEP_VAR_LIST) * len(HORIZON_LIST) * len(WINDOW_LIST) * len(CONTROL_GROUPS)
count = 0

print(f"--- Starting Batch Rolling Regression ---")
print(f"Total combinations to run: {total_runs}")
print(f"Aligning all plots to start after {MAX_WINDOW_SIZE} days.")

for dep_var in DEP_VAR_LIST:
    for horizon in HORIZON_LIST:
        for window in WINDOW_LIST:
            for control_name, control_vars in CONTROL_GROUPS:
                
                count += 1
                run_id = f"Y: {dep_var} | H: {horizon} | W: {window} | Ctrl: {control_name}"
                print(f"[{count}/{total_runs}] Running: {run_id}")

                # A. Build Data
                df_run = build_panel(dep_var, horizon, control_vars, SELECTED_COUNTRIES)
                
                if df_run is None or df_run.empty:
                    print("   -> Skipped (No Data)")
                    continue

                # B. Run Rolling Regression
                unique_dates = df_run.index.get_level_values(1).unique().sort_values()
                
                # --- FIX 1: Determine Common Start Date ---
                # We find the date corresponding to the largest window size (MAX_WINDOW_SIZE)
                # This ensures all graphs (even those with small windows) start at the same visual point.
                if len(unique_dates) > MAX_WINDOW_SIZE:
                    common_start_date = unique_dates[MAX_WINDOW_SIZE]
                else:
                    print("   -> Not enough data for max window.")
                    continue
                # ------------------------------------------

                x_vars = ['X_Predictor'] + [c for c in control_vars if c in df_run.columns]
                
                betas, dates, upper, lower = [], [], [], []
                
                # Start loop
                for i in range(window, len(unique_dates)):
                    current_date = unique_dates[i]
                    window_dates = unique_dates[i-window : i]
                    
                    mask = df_run.index.get_level_values(1).isin(window_dates)
                    window_data = df_run[mask]

                    if len(window_data) < (window * len(SELECTED_COUNTRIES) * 0.2):
                        continue

                    try:
                        mod = PanelOLS(window_data['Y_Target'], window_data[x_vars], entity_effects=True)
                        res = mod.fit(cov_type='robust') 
                        
                        b = res.params['X_Predictor']
                        se = res.std_errors['X_Predictor']
                        
                        betas.append(b)
                        upper.append(b + 1.645*se) # 90% CI
                        lower.append(b - 1.645*se)
                        dates.append(current_date)
                    except:
                        continue

                # C. Plot and Save
                if betas:
                    # Create DataFrame for easier slicing
                    df_results = pd.DataFrame({
                        'Beta': betas,
                        'Upper': upper,
                        'Lower': lower
                    }, index=dates)

                    # --- FIX 1 Applied: Trim data to start at common date ---
                    df_plot = df_results.loc[common_start_date:]
                    
                    if df_plot.empty:
                        print("   -> Data empty after trimming to common start date.")
                        continue

                    fig, ax = plt.subplots(figsize=(12, 6))
                    
                    # Data Series
                    ax.fill_between(df_plot.index, df_plot['Lower'], df_plot['Upper'], color='steelblue', alpha=0.2, label='90% CI')
                    ax.plot(df_plot.index, df_plot['Beta'], color='black', lw=2, label=f'Beta (Export -> {dep_var})')
                    
                    # Aesthetics
                    ax.axhline(0, color='red', linestyle='--', linewidth=1)
                    ax.axvline(pd.to_datetime("2020-03-01"), color="blue", linestyle=":", lw=1.5, label="COVID")
                    ax.axvline(pd.to_datetime("2022-02-24"), color="green", linestyle=":", lw=1.5, label="Ukraine War")
                    
                    # --- FIX 2: Explicit X-Axis Limits ---
                    # This removes the whitespace on the left and right
                    ax.set_xlim(left=df_plot.index.min(), right=df_plot.index.max())
                    
                    title_str = f"Rolling Beta: Export Price -> {dep_var}\nHorizon: {horizon} | Window: {window} | Controls: {control_name}"
                    #ax.set_title(title_str, fontsize=14)
                    #ax.set_ylabel("Coefficient", fontsize=12)
                    ax.grid(True, linestyle='--', alpha=0.6)
                    ax.legend(loc='upper left')

                    # Filename
                    fname = f"Rolling_Beta_{dep_var}_h{horizon}_w{window}_{control_name}.png"
                    plt.savefig(os.path.join(OUTPUT_FOLDER, fname), bbox_inches='tight')
                    plt.close(fig)
                    print(f"   -> Saved: {fname}")
                else:
                    print("   -> No regressions converged.")

print("\n--- Batch Processing Complete ---")

# Forecast Regression - Dynamic Model Averaging

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import itertools
import statsmodels.api as sm
from tqdm import tqdm 
import os
import warnings

warnings.filterwarnings('ignore')

# --- 1. BATCH CONFIGURATION ---
START_DATE = "2010-01-01"
END_DATE = "2025-06-30"

# Define output folder
OUTPUT_FOLDER = "Results_BMA"
if not os.path.exists(OUTPUT_FOLDER): os.makedirs(OUTPUT_FOLDER)

# --- GRID SEARCH SETTINGS ---
# 1. Country Groups
commodity_exporters = ["Australia", "Brazil", "Canada", "Chile", "Colombia", "Mexico", "Peru", "Norway", "South Africa"]
COUNTRY_CONFIGS = [
    ("All_Countries", None),
    ("Commodity_Exporters", commodity_exporters)
]

# 2. Dependent Variables
DEP_VAR_LIST = ['log_neer', 'log_fx']

# 3. Horizons (Days)
HORIZON_LIST = [1, 5, 22] # e.g., 1 month. Can add [5, 60] etc.

# 4. Window Sizes (Days)
WINDOW_LIST = [100, 252, 504, 1016] # e.g. 4 years. Can add [500] etc.

# Variables
FIXED_REGRESSORS = ['log_export_shock'] 
CANDIDATE_VARS = ['inter_gpr', 'inter_vix', 'inter_roro', 'inter_tpu', 'inter_ird_usa']

# Country Mapping
country_name_mapping = {
    "Euro area": "Germany", 
    "USA": "United States",
}

# --- 2. HELPER FUNCTIONS ---

def compute_bma_weights(y, X_fixed, X_candidates):
    """ Calculates Posterior Inclusion Probabilities (PIP) using BIC approximation. """
    max_vars = 3 
    combinations = []
    for r in range(1, max_vars + 1):
        combinations.extend(itertools.combinations(X_candidates.columns, r))
    combinations.append(()) 

    models_bic = []
    models_vars = []
    
    for combo in combinations:
        current_vars = list(combo)
        if not X_fixed.empty:
            X_subset = pd.concat([X_fixed, X_candidates[current_vars]], axis=1)
        else:
            X_subset = X_candidates[current_vars]
        
        try:
            # No constant because data is demeaned (Fixed Effects)
            model = sm.OLS(y, X_subset).fit()
            models_bic.append(-0.5 * model.bic)
            models_vars.append(current_vars)
        except:
            continue

    if not models_bic: return {v: 0.0 for v in X_candidates.columns}

    bics = np.array(models_bic)
    bics = bics - np.max(bics) 
    model_weights = np.exp(bics)
    model_weights = model_weights / np.sum(model_weights)
    
    pips = {v: 0.0 for v in X_candidates.columns}
    for vars_in_model, weight in zip(models_vars, model_weights):
        for v in vars_in_model:
            pips[v] += weight
    return pips

def build_bma_panel(dep_var, horizon, countries_list):
    """ 
    Constructs the interaction dataset and performs demeaning (FE).
    Returns a dataframe ready for pooled OLS.
    """
    panel_data = []
    available = df_neer.columns
    
    if countries_list:
        targets = [c for c in countries_list if c in available]
    else:
        targets = [c for c in available if c not in ["USA", "United States"]]

    for country in targets:
        if dep_var == 'log_fx' and country in ["USA", "United States"]: continue
        
        try:
            temp = pd.DataFrame(index=df_neer.index)
            
            # Y & X
            if dep_var == 'log_neer': y_level = np.log(df_neer[country])
            elif dep_var == 'log_fx': 
                if country in df_fx.columns: y_level = np.log(df_fx[country])
                else: continue
            
            x_shock = np.log(df_export[country]).diff()
            
            # Uncertainty Levels (Using levels for interaction)
            u_gpr = np.log(df_gpr.iloc[:,0])
            u_vix = np.log(df_vix.iloc[:,0])
            u_roro = df_roro.iloc[:,0]
            u_tpu = np.log(df_tpu.iloc[:,0])
            
            c_map = country_name_mapping.get(country, country)
            if c_map in df_ird_usa.columns: u_ird = df_ird_usa[c_map]
            else: continue

            # Interactions: Shock * Level
            temp['inter_gpr'] = x_shock * u_gpr
            temp['inter_vix'] = x_shock * u_vix
            temp['inter_roro'] = x_shock * u_roro
            temp['inter_tpu'] = x_shock * u_tpu
            temp['inter_ird_usa'] = x_shock * u_ird
            
            temp['log_export_shock'] = x_shock
            temp['Target'] = y_level.shift(-horizon) - y_level # Future Return
            temp['Country'] = country
            
            temp = temp.loc[START_DATE:END_DATE]
            temp.replace([np.inf, -np.inf], np.nan, inplace=True)
            
            panel_data.append(temp)
        except KeyError:
            continue

    if not panel_data: return None

    df_panel = pd.concat(panel_data)
    df_panel.dropna(inplace=True)
    
    # Demean by Country (Fixed Effects transformation)
    cols_to_demean = ['Target'] + FIXED_REGRESSORS + CANDIDATE_VARS
    df_demeaned = df_panel.groupby('Country')[cols_to_demean].transform(lambda x: x - x.mean())
    df_demeaned.index = df_panel.index
    
    return df_demeaned

# --- 3. MASTER LOOP ---
total_runs = len(COUNTRY_CONFIGS) * len(DEP_VAR_LIST) * len(HORIZON_LIST) * len(WINDOW_LIST)
count = 0

print(f"--- Starting Batch BMA Analysis ---")
print(f"Total Runs: {total_runs}")

for panel_name, selected_countries in COUNTRY_CONFIGS:
    for dep_var in DEP_VAR_LIST:
        for horizon in HORIZON_LIST:
            for window in WINDOW_LIST:
                
                count += 1
                run_id = f"{panel_name} | Y: {dep_var} | H: {horizon} | W: {window}"
                print(f"\n[{count}/{total_runs}] Processing: {run_id}")

                # A. Build Data
                df_run = build_bma_panel(dep_var, horizon, selected_countries)
                
                if df_run is None or df_run.empty:
                    print("   -> Skipped (No Data)")
                    continue

                # B. Run Rolling BMA
                unique_dates = df_run.index.unique().sort_values()
                results_pip = []
                date_axis = []
                
                # Step size for speed (e.g., update every 22 days / 1 month)
                step_size = 22 
                
                # Use tqdm for progress bar within the loop
                for i in tqdm(range(window, len(unique_dates), step_size), desc="   -> Rolling BMA"):
                    current_date = unique_dates[i]
                    window_dates = unique_dates[i-window : i]
                    
                    mask = df_run.index.isin(window_dates)
                    data_slice = df_run[mask]
                    
                    # Min observations check (e.g., 20% of theoretical max)
                    if len(data_slice) < 100: continue
                    
                    pips = compute_bma_weights(
                        data_slice['Target'], 
                        data_slice[FIXED_REGRESSORS], 
                        data_slice[CANDIDATE_VARS]
                    )
                    results_pip.append(pips)
                    date_axis.append(current_date)

                # C. Plot and Save
                if results_pip:
                    df_results = pd.DataFrame(results_pip, index=date_axis)
                    df_results = df_results.rolling(window=6).mean() # Smooth visuals

                    fig, ax = plt.subplots(figsize=(12, 7))
                    df_results.plot.area(ax=ax, alpha=0.85, colormap='viridis') 

                    ax.set_title(f"Drivers of Commodity-Currency Sensitivity (PIP)\nPanel: {panel_name} | Y: {dep_var} | H: {horizon} | W: {window}", fontsize=14)
                    ax.set_ylabel("Posterior Inclusion Probability", fontsize=12)
                    ax.set_ylim(0, None)
                    
                    # Events
                    ax.axvline(pd.to_datetime("2020-03-01"), color="white", linestyle="--", label="COVID")
                    ax.axvline(pd.to_datetime("2022-02-24"), color="white", linestyle=":", label="Ukraine")
                    
                    plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), title="Interaction Terms")
                    plt.tight_layout()

                    # Filename
                    fname = f"BMA_{panel_name}_{dep_var}_h{horizon}_w{window}.png"
                    plt.savefig(os.path.join(OUTPUT_FOLDER, fname), bbox_inches='tight')
                    plt.close(fig)
                    print(f"   -> Saved plot: {fname}")
                else:
                    print("   -> No results generated.")

print("\n--- BMA Batch Complete ---")

# BMA Plot

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

# --- CONFIGURATION ---
INPUT_FOLDER = "Results/BMA"
HORIZONS = [1, 5, 22]
VARIABLES_TO_PLOT = ["Commodity Export","IRD","VIX","GPR","RoRo Spreads","RoRo Equities","RoRo Liquidity","Surprise"]
COMBINED_VARIABLES = ["VIX","RoRo Equities"]

# Create output folder
OUTPUT_FOLDER = "Results/BMA_Plots"
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

def plot_bma_pips(horizon):
    filename = f"Mod3_h{horizon}.xlsx"
    filepath = os.path.join(INPUT_FOLDER, filename)
    
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return

    # Load Data
    df = pd.read_excel(filepath)
    # Ensure Date index
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
        df.set_index('date', inplace=True)
    elif 'Date' in df.columns:
        df['Date'] = pd.to_datetime(df['Date'])
        df.set_index('Date', inplace=True)
    
    # Calculate limits for tight layout
    min_date = df.index.min()
    max_date = df.index.max()

    print(f"--- Plotting Horizon h={horizon} ---")

    # 1. Plot Individual Variables
    for var in VARIABLES_TO_PLOT:
        if var in df.columns:
            plt.figure(figsize=(10, 4))
            
            # Changed color to 'steelblue' and increased linewidth slightly
            plt.plot(df.index, df[var], color='steelblue', linewidth=2)
            
            # Styling
            plt.title(f"{var} (h={horizon})", fontsize=14)
            plt.ylabel("Posterior Inclusion Probability", fontsize=10)
            plt.ylim(-0.05, 1.05)
            
            # --- FIX: Tight X-Axis ---
            plt.xlim(min_date, max_date)
            
            plt.grid(True, linestyle='--', alpha=0.6)
            
            # Save
            save_path = os.path.join(OUTPUT_FOLDER, f"PIP_h{horizon}_{var}.png")
            plt.savefig(save_path, bbox_inches='tight')
            plt.close()
            print(f"Saved: {save_path}")

    # 2. Plot Combined Variables (Spreads & Equities)
    if all(col in df.columns for col in COMBINED_VARIABLES):
        plt.figure(figsize=(10, 4))
        
        # Plot Spreads vs Equities
        plt.plot(df.index, df["VIX"], color="steelblue", linewidth=1.5, label='VIX') # Standard Matplotlib Blue
        plt.plot(df.index, df['RoRo Equities'], color="#e06c1f", linewidth=1.5, label='RoRo Equities')   # Standard Matplotlib Red
        
        # Styling
        plt.title(f"VIX vs RoRo Equities (h={horizon})", fontsize=14)
        plt.ylabel("Posterior Inclusion Probability", fontsize=10)
        plt.ylim(-0.05, 1.15) # Room for legend
        
        # --- FIX: Tight X-Axis ---
        plt.xlim(min_date, max_date)

        plt.grid(True, linestyle='--', alpha=0.6)
        
        # Legend outside/bottom
        plt.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15),
                   fancybox=True, shadow=False, ncol=2)
        
        # Save
        save_path = os.path.join(OUTPUT_FOLDER, f"PIP_h{horizon}_VIX_vs_RoRO_Equities.png")
        plt.savefig(save_path, bbox_inches='tight')
        plt.close()
        print(f"Saved: {save_path}")

# --- MAIN LOOP ---
for h in HORIZONS:
    plot_bma_pips(h)

print("\nAll BMA plots generated successfully.")

# BMA Plot Stacked

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import numpy as np

# --- CONFIGURATION ---
INPUT_FOLDER = "Results/BMA"
HORIZONS = [1, 5, 22]

# Variables you want to include in the stack
VARIABLES_TO_PLOT = [
    "Commodity Export", "IRD", "VIX", "GPR", 
    "RoRo Spreads", "RoRo Equities", "RoRo Liquidity", "Surprise"
]

# Create output folder
OUTPUT_FOLDER = "Results/BMA_Stacked_Plots_NEER"
if not os.path.exists(OUTPUT_FOLDER):
    os.makedirs(OUTPUT_FOLDER)

def plot_stacked_bma(horizon):
    filename = f"Mod3_h{horizon}.xlsx"
    filepath = os.path.join(INPUT_FOLDER, filename)
    
    if not os.path.exists(filepath):
        print(f"File not found: {filepath}")
        return

    # 1. Load and Clean Data
    df = pd.read_excel(filepath)
    if 'date' in df.columns:
        df.rename(columns={'date': 'Date'}, inplace=True)
    
    df['Date'] = pd.to_datetime(df['Date'])
    df.set_index('Date', inplace=True)
    
    # Filter only the variables we have in the dataframe
    available_vars = [v for v in VARIABLES_TO_PLOT if v in df.columns]
    plot_df = df[available_vars].fillna(0) # Stackplot doesn't like NaNs

    if plot_df.empty:
        print(f"No variables found for horizon {horizon}")
        return

    # 2. Setup Plot
    fig, ax = plt.subplots(figsize=(12, 7))
    
    # Create a nice color palette
    # 'tab10' or 'Set3' are good for categorical labels
    colors = plt.cm.tab10(np.linspace(0, 1, len(available_vars)))

    # 3. Create Stacked Area Plot
    # .T (transpose) is needed because stackplot expects (variables, time)
    stacks = ax.stackplot(plot_df.index, plot_df.values.T, 
                          labels=available_vars, 
                          colors=colors, 
                          alpha=0.85,
                          edgecolor='white', # Adds a thin line between variables for clarity
                          linewidth=0.2)

    # 4. Styling
    #ax.set_title(f"Model Composition: Posterior Inclusion Probabilities (h={horizon})", fontsize=16, fontweight='bold', pad=20)
    #ax.set_ylabel("PIP", fontsize=12)
    ax.set_xlabel("Date", fontsize=12)
    
    # X-Axis Tightness
    ax.set_xlim(plot_df.index.min(), plot_df.index.max())
    
    # Grid (only horizontal for stacked plots usually looks cleaner)
    ax.grid(axis='y', linestyle='--', alpha=0.3)

    # 5. Legend
    # Placing the legend outside to the right so it doesn't cover the data
    n_cols = 4 if len(available_vars) > 4 else len(available_vars)
    
    ax.legend(loc='upper center', 
              bbox_to_anchor=(0.5, -0.12), 
              ncol=n_cols, 
              fontsize=10, 
              frameon=False)

    # Tight layout to make room for the legend
    plt.tight_layout()

    # 6. Save
    save_path = os.path.join(OUTPUT_FOLDER, f"Stacked_PIP_h{horizon}.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Saved Stacked Plot: {save_path}")

# --- MAIN LOOP ---
print(f"--- Generating Stacked BMA Plots ---")
for h in HORIZONS:
    plot_stacked_bma(h)

print("\nAll stacked BMA plots generated successfully.")

# END